# Eleven anomalies of a synthetic traffic dataset — what TrafficFlowBench's generator does and does not do

TrafficFlowBench is synthetic. That is not a complaint: it is the single most useful fact about the competition, because a generator
has rules, and rules can be measured. This notebook is the list of places where the released data behaves **unlike a real freeway**,
each one with the test that shows it and the number it gives, recomputed live below from one corridor-month (I-10 westbound in
District 7, February 2031: 107 links, 49 km, 8,064 five-minute steps, 115 ramps).

Why it matters: every "physics" feature we built on real-road intuition — ramp inflow, travel time, shock waves, truck share,
conservation of vehicles — scored exactly zero locally, and every one of these anomalies explains one of those zeros. The companion
notebook [*The physics of a synthetic jam*](https://www.kaggle.com/code/plarmuseau/trafficflowbench-the-physics-of-a-synthetic-jam)
covers the jam dynamics; this one covers the generator's bookkeeping. The measurement log behind both is a single append-only file
(`NOTA.md`, ~1,900 lines); the numbers quoted in the text come from it, the code cells reproduce them.

1. Ramps do not enter the mainline balance
2. A ramp switches off when the mainline next to it jams
3. The day profile is synchronous along 49 km (no travel time)
4. Cumulative in minus cumulative out swings by ±2,000 vehicles
5. Free-flow speed is one constant per link, with a cap at 120.7 km/h
6. There is no traffic composition (no truck share, no hour or weekday effect on free speed)
7. The noise is white and multiplicative: 2.8 % on flow, 1.2 km/h on speed, independent
8. There are no demand pulses: a smooth day curve plus white noise
9. A jam has a fixed head, a straight tail, and no conservation inside it
10. The conservation score is zero for the truth itself under the public topology
11. The masks are coin flips with a short memory
12. What it means for a model (scoreboard)

In [ ]:
import os, glob, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')
def find_root():
    if os.environ.get('TFB_ROOT'): return Path(os.environ['TFB_ROOT'])
    for depth in range(1, 7):
        for p in Path('/kaggle/input').glob('/'.join(['*'] * depth) + '/corridors'):
            if (p / 'D7_I10_W').exists(): return p.parent
    raise FileNotFoundError('competition data not found under /kaggle/input')
ROOT = find_root(); PANEL = 'D7_I10_W'; D = ROOT / 'corridors' / PANEL; print('data at', ROOT)
topo = pd.read_csv(D / 'network' / 'lwr_mainline_topology.csv').sort_values('order_index').reset_index(drop=True)
fd = pd.read_csv(D / 'network' / 'fd_parameters.csv').set_index('link_id')
LINKS = topo.link_id.tolist(); L = len(LINKS); LANES = topo.lanes.to_numpy(float); VF = fd.free_speed_kmh.reindex(LINKS).to_numpy()
# traffic runs from high order_index to low: order L-1 is the entrance, order 0 the exit. x = km from the entrance.
X = np.cumsum(topo.length_km.to_numpy()[::-1])[::-1] - topo.length_km.to_numpy()      # km from the entrance: order L-1 → 0, order 0 → 48.7
split = lambda s: [] if pd.isna(s) else s.split(';')
ON = [split(s) for s in topo.on_ramp_link_ids]; OFF = [split(s) for s in topo.off_ramp_link_ids]
def load_month(ym='2031-02', layer='train'):
    V, Q, R = [], [], []
    for f in sorted(glob.glob(str(D / layer / 'mainline_states' / f'year_month={ym}' / '*.parquet'))):
        d = pd.read_parquet(f, columns=['timestamp', 'link_id', 'speed_kmh', 'flow_vph', 'is_observed']); d.loc[d.is_observed != 1, ['speed_kmh', 'flow_vph']] = np.nan
        V.append(d.pivot(index='timestamp', columns='link_id', values='speed_kmh').reindex(columns=LINKS)); Q.append(d.pivot(index='timestamp', columns='link_id', values='flow_vph').reindex(columns=LINKS))
        r = pd.read_parquet(f.replace('mainline_states', 'ramp_states').replace('synthetic_mainline', 'synthetic_ramp'), columns=['timestamp', 'ramp_link_id', 'flow_vph', 'is_observed']); r.loc[r.is_observed != 1, 'flow_vph'] = np.nan
        R.append(r.pivot(index='timestamp', columns='ramp_link_id', values='flow_vph'))
    V = pd.concat(V); Q = pd.concat(Q); R = pd.concat(R).reindex(V.index); t = pd.to_datetime(V.index, utc=True)
    return V, Q, R, t
V, Q, R, T = load_month(); N = len(T); HOUR = T.hour.to_numpy() + T.minute.to_numpy() / 60; DAY = T.dayofyear.to_numpy() - T.dayofyear.min(); WEEKEND = T.dayofweek.to_numpy() >= 5
print(f'{PANEL}: {L} links, {X.max():.1f} km, {N} steps, {len(np.unique(DAY))} days; speed observed {np.isfinite(V.to_numpy()).mean():.3f}; ramps {R.shape[1]} observed {np.isfinite(R.to_numpy()).mean():.3f}')

## 1. Ramps do not enter the mainline balance

On a real freeway the flow at the exit equals the flow at the entrance plus what the on-ramps add minus what the off-ramps take.
Here the mainline balances **on its own**: for one day the entrance passes 90,724 vehicles and the exit 90,260 (+0.5 %), while the
ramps add a net +13,654 — include them and the day closes at **+15.6 %**. Along the road the daily-mean flow is flat to 0.5 % from
link to link, whereas a 4-km interchange with 13 on-ramps and 12 off-ramps should add ~800 veh/h; regressing the link-to-link flow
gain on the net ramp flow joining there gives a slope of 0.05 where physics says 1.00.
The ramp files are a separate draw next to the mainline. Every ramp feature we built for Task 1 scored zero for this reason.

In [ ]:
fill = lambda s: s.interpolate(limit_direction='both')
on_all = fill(R.reindex(columns=sum(ON, [])).sum(axis=1, min_count=1)).fillna(0).to_numpy(); off_all = fill(R.reindex(columns=sum(OFF, [])).sum(axis=1, min_count=1)).fillna(0).to_numpy()
qin = fill(Q.iloc[:, -1]).to_numpy(); qout = fill(Q.iloc[:, 0]).to_numpy()
rows = []
for d in np.unique(DAY):
    s = DAY == d; rows.append(dict(day=str(T[s][0].date()), entrance=qin[s].sum() / 12, exit=qout[s].sum() / 12, on_ramps=on_all[s].sum() / 12, off_ramps=off_all[s].sum() / 12))
tab = pd.DataFrame(rows); tab['closure_without_ramps'] = (tab.entrance - tab.exit) / tab.exit; tab['closure_with_ramps'] = (tab.entrance + tab.on_ramps - tab.off_ramps - tab.exit) / tab.exit
print(tab.round(3).head(7).to_string(index=False)); print(f'\n28 days: closure without ramps mean {tab.closure_without_ramps.mean():+.3%} (sd {tab.closure_without_ramps.std():.3%}); with ramps {tab.closure_with_ramps.mean():+.3%}')
# along the road: daily-mean mainline flow per link vs entrance + cumulative net ramp flow
free = (HOUR >= 10) & (HOUR < 15)
qbar = np.nanmean(Q.to_numpy()[free], 0); net = np.array([np.nanmean(fill(R.reindex(columns=ON[k]).sum(axis=1, min_count=1)).fillna(0).to_numpy()[free]) - np.nanmean(fill(R.reindex(columns=OFF[k]).sum(axis=1, min_count=1)).fillna(0).to_numpy()[free]) for k in range(L)])
expected = qbar[-1] + np.cumsum(net[::-1])[::-1] - net          # entrance flow + net ramps joined upstream of each link
plt.figure(figsize=(11, 4)); plt.plot(X, qbar, 'o-', ms=3, label='measured mainline flow per link (10–15 h mean)'); plt.plot(X, expected, '--', label='entrance flow + cumulative (on − off) ramps upstream')
plt.xlabel('km from entrance'); plt.ylabel('veh/h'); plt.title(f'{PANEL} February: the mainline ignores its ramps (link-to-link sd of the measured line {np.std(np.diff(qbar)):.0f} veh/h)'); plt.legend(); plt.grid(alpha=.3); plt.show()
gain = qbar[:-1] - qbar[1:]          # flow gained between the upstream link k+1 and the downstream link k
print(f'slope of that gain on the net ramp flow joining at link k: {np.polyfit(net[:-1], gain, 1)[0]:+.2f} veh/h per veh/h (physics: 1.00); corr {np.corrcoef(net[:-1], gain)[0, 1]:+.2f}')

## 2. A ramp switches off when the mainline next to it jams

The one place the ramps *do* react to the mainline is binary: when the nearest mainline link is in a jam (speed < 0.6 × free speed)
the on-ramp flow drops to **0.2 × its hour profile** (0.07 at the ramp we first saw it on), 81 % of the steps exactly zero, in the same
5-minute step as the jam, and recovers in the same step the jam clears; off-ramps are untouched (1.0–1.1 × profile). That makes the ramp files a jam sensor. The released ramp files cover the Task-2 forecast horizon, so this
sensor is visible after the forecast origin T; using it is a leak under the organizers' clarified rule (nothing after T). We measured it
locally only: +0.038 IoU for ongoing jams from the horizon rows, versus +0.008 from the legal history rows.

In [ ]:
amap = pd.read_csv(D / 'network' / 'ramp_attachment_map.csv'); amap = amap[amap.nearest_mainline_link_id.isin(LINKS) & amap.ramp_link_id.isin(R.columns)]
pos = {l: i for i, l in enumerate(LINKS)}; jam = (V.to_numpy() < 0.6 * VF[None, :])
hr = T.hour.to_numpy(); out = []
for r in amap.itertuples():
    f = R[r.ramp_link_id].to_numpy(); prof = pd.Series(f).groupby(hr).transform('mean').to_numpy(); ok = np.isfinite(f) & (prof > 5); j = jam[:, pos[r.nearest_mainline_link_id]] & ok
    if j.sum() < 12: continue
    out.append(dict(ramp=r.ramp_link_id, type=r.ramp_type, jam_steps=int(j.sum()), ratio_in_jam=np.mean(f[j] / prof[j]), ratio_free=np.mean(f[ok & ~j] / prof[ok & ~j]), zero_share_in_jam=np.mean(f[j] == 0), zero_share_free=np.mean(f[ok & ~j] == 0)))
o = pd.DataFrame(out); print(o.groupby('type')[['ratio_in_jam', 'ratio_free', 'zero_share_in_jam', 'zero_share_free']].median().round(3)); print(f'{len(o)} ramps with >= 1 h of jam next to them in February')
w = o.sort_values('jam_steps').iloc[-1]; f = R[w.ramp]; j = jam[:, pos[amap.set_index('ramp_link_id').nearest_mainline_link_id[w.ramp]]]; d = DAY[np.argmax(pd.Series(j).groupby(DAY).sum().reindex(DAY).to_numpy())]; s = DAY == d
fig, ax = plt.subplots(figsize=(11, 3.5)); ax.plot(HOUR[s], f.to_numpy()[s], label=f'{w.ramp} ({w.type}) flow'); ax.fill_between(HOUR[s], 0, np.nanmax(f.to_numpy()[s]), where=j[s], color='red', alpha=.2, label='jam on the nearest mainline link'); ax.set_xlabel('hour'); ax.set_ylabel('veh/h'); ax.legend(); ax.set_title('the ramp is a jam sensor'); plt.show()

## 3. The day profile is synchronous along 49 km

Vehicles need 26 minutes to drive the corridor at 114 km/h. The daily demand curve at the exit should therefore lag the one at the
entrance by about five 5-minute steps. It does not: the correlation between entrance and exit flow is highest at **lag 0** (0.964) and
falls monotonically with every shift. The generator imposes its demand curve on the whole corridor at once; only the small 45–55 min
pulses of section 8 actually travel. A model does not need to know the travel time, and features shifted by distance/speed gained nothing.

In [ ]:
sm = lambda a, w=5: pd.Series(a).rolling(w, center=True, min_periods=1).mean().to_numpy()
lags = range(0, 10); day = DAY == 2; a, b = sm(qin[day]), sm(qout[day])
cc = [np.corrcoef(a[:len(a) - k], b[k:])[0, 1] for k in lags]
tt = X.max() / np.nanmean(V.to_numpy()) * 60
print('corr(entrance, exit) per lag of 5 min:', np.round(cc, 3)); print(f'best lag {5 * int(np.argmax(cc))} min; travel time at the mean speed {np.nanmean(V.to_numpy()):.0f} km/h = {tt:.0f} min')
fig, ax = plt.subplots(figsize=(11, 3.5)); ax.plot(HOUR[day], a, label='entrance (25-min smoothed)'); ax.plot(HOUR[day], b, label='exit'); ax.plot(HOUR[day][5:], a[:-5], '--', alpha=.6, label='entrance shifted by the 26-min travel time (worse fit)'); ax.legend(); ax.set_xlabel('hour'); ax.set_ylabel('veh/h'); ax.set_title('the profile does not travel'); plt.show()

## 4. Cumulative in minus cumulative out swings by ±2,000 vehicles

Integrate the entrance flow and the exit flow over a day. On a closed road the difference is the number of vehicles between the two
detectors: with ~15 veh/km/lane on 49 km × 3–4 lanes that is about 2,500 vehicles, and it can never go negative. Here N_in − N_out
drops to **−2,200** around 13 h (the exit has released more vehicles than ever entered) and climbs to +780 in the evening. Each link
carries its own day shape with no upstream cause — the "fingerprint" of a link is a profile, not a vehicle population.

In [ ]:
Nin = np.cumsum(qin[day]) / 12; Nout = np.cumsum(qout[day]) / 12; gap = Nin - Nout
fig, ax = plt.subplots(figsize=(11, 3.5)); ax.plot(HOUR[day], gap, color='k'); ax.axhline(0, color='grey', lw=.5); ax.set_xlabel('hour'); ax.set_ylabel('N_in − N_out (vehicles)')
ax.set_title(f'vehicles "on the road" by the counts: min {gap.min():.0f}, max {gap.max():.0f}; physically ~{np.nanmean(Q.to_numpy()[day] / V.to_numpy()[day]) * X.max():.0f} vehicles, never negative'); plt.show()

## 5. Free-flow speed is one constant per link, with a cap at 120.7 km/h

In free flow every link runs at its own level between 101 and 121 km/h. That level does not move: the between-day standard deviation
of a link's free-flow median is **0.10 km/h** and the levels of two halves of the month correlate 1.000. The level is not the
`free_speed_kmh` of `fd_parameters.csv` (105–118) but a hidden constant per link, and links whose constant sits above the generator's
clip at 120.7 km/h have their noise cut off: their residual standard deviation is 0.65 km/h on average, down to 0.10, against 1.3 elsewhere. Those are the
"white bands" one sees on any error map, and the noise level per link is itself a fixed property (first vs second half of the month: 0.989).

In [ ]:
v = V.to_numpy(); freec = v > 0.8 * VF[None, :]
lvl = np.array([[np.nanmedian(np.where(freec[DAY == d, k], v[DAY == d, k], np.nan)) for k in range(L)] for d in np.unique(DAY)])
level = np.nanmedian(lvl, 0); sd_days = np.nanstd(lvl, 0)
dev = np.where(freec, v - level[None, :], np.nan); sd1 = np.nanstd(dev[DAY < 14], 0); sd2 = np.nanstd(dev[DAY >= 14], 0)
print(f'free-flow level per link: {level.min():.1f} … {level.max():.1f} km/h; between-day sd of the level: median {np.median(sd_days):.2f} km/h; corr(level 1st half, 2nd half) = {np.corrcoef(np.nanmedian(lvl[:14], 0), np.nanmedian(lvl[14:], 0))[0, 1]:.3f}')
print(f'noise sd per link: {np.nanmin(sd1):.2f} … {np.nanmax(sd1):.2f}; corr(1st half, 2nd half) = {np.corrcoef(sd1, sd2)[0, 1]:.3f}; links above 120 km/h: {np.sum(level > 120)} with sd {np.nanmean(sd1[level > 120]):.2f} vs {np.nanmean(sd1[level <= 120]):.2f} elsewhere')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); ax[0].plot(X, level, 'o-', ms=3); ax[0].plot(X, VF, 's', ms=3, alpha=.5, label='fd_parameters free_speed_kmh'); ax[0].axhline(120.7, color='r', lw=.8, label='cap 120.7'); ax[0].set_xlabel('km'); ax[0].set_ylabel('km/h'); ax[0].set_title('free-flow level per link (February median)'); ax[0].legend(fontsize=8)
ax[1].scatter(level, sd1, s=12); ax[1].set_xlabel('free-flow level km/h'); ax[1].set_ylabel('residual sd km/h'); ax[1].set_title('the clip removes the noise above 120 km/h'); plt.show()

## 6. There is no traffic composition

Real mean speeds are a mixture: cars at the limit, trucks at 90–100 km/h, and the truck share changes with the hour, the flow and the
weekday. If this generator drew a population, the free-flow deviation from the link level would fall with the flow (more trucks in the
way), be lower at night (higher truck share) and differ on weekends. All three effects are zero: hour effect −0.01 to −0.11 km/h
flat, slope vs relative flow **+0.02 km/h per 100 % of capacity** (correlation 0.003), weekday −0.05 vs weekend −0.07, between-day sd
0.04. The distribution around the level is unimodal with skew −0.36 (the clip at 120.7 makes the left tail). Composition features have nothing to model.

In [ ]:
from scipy.stats import skew
q = Q.to_numpy(); rel = q / fd.capacity_vph.reindex(LINKS).to_numpy()[None, :]
dv = dev[freec]; h = np.repeat(HOUR[:, None], L, 1)[freec]; rq = rel[freec]; we = np.repeat(WEEKEND[:, None], L, 1)[freec]; ok = np.isfinite(dv) & np.isfinite(rq)
by_hour = pd.Series(dv[ok]).groupby(h[ok].astype(int)).mean(); print('mean deviation from the link level by hour of day (km/h):', np.round(by_hour.to_numpy(), 2))
sl = np.polyfit(rq[ok], dv[ok], 1)[0]; print(f'slope vs relative flow: {sl:+.3f} km/h per 100 % of capacity (corr {np.corrcoef(rq[ok], dv[ok])[0, 1]:+.3f}); weekday {dv[ok & ~we].mean():+.2f} vs weekend {dv[ok & we].mean():+.2f}; skew {skew(dv[ok]):+.2f}')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); ax[0].hist(dv[ok], bins=120, range=(-8, 5)); ax[0].set_title('deviation from the link level, free cells: one mode'); ax[0].set_xlabel('km/h')
b = pd.cut(rq[ok], np.arange(0, 1.01, .1)); ax[1].plot(pd.Series(dv[ok]).groupby(b, observed=True).mean().to_numpy(), 'o-'); ax[1].set_xlabel('relative flow bin (0.1 steps)'); ax[1].set_ylabel('mean deviation km/h'); ax[1].set_ylim(-2, 2); ax[1].set_title('no slope with flow'); plt.show()

## 7. The noise is white, multiplicative and independent

Two detectors 300 m apart with no ramp between them count the same vehicles. The relative difference of their flows has sd **0.033–0.035**
whether 80 vehicles pass per 5 minutes (night) or 360 (day); Poisson counting would give 0.16 at night and 0.075 by day. So the generator
adds ±2.3 % multiplicative noise per detector, i.i.d. — the Task-1 flow floor is 2.8 % of the flow per cell, and 96 % of our flow error
is exactly that, with no link bias. Speed noise is 1.1–1.3 km/h, independent of the flow noise (correlation 0.007), and the residual
spectrum is flat. That is the ceiling of the state score: nothing in the data predicts the sign of the next cell's noise
(a lazypredict sweep of 30 regressors on 40 × 5 raw windows was won by the dummy).

In [ ]:
pairs = [(k, k + 1) for k in range(L - 1) if not (ON[k] or OFF[k] or ON[k + 1] or OFF[k + 1]) and LANES[k] == LANES[k + 1] and abs(X[k] - X[k + 1]) < 0.5]
rd = np.concatenate([(q[:, a] - q[:, b]) / q[:, b] for a, b in pairs]); cnt = np.concatenate([q[:, b] / 12 for a, b in pairs]); ok = np.isfinite(rd) & np.isfinite(cnt) & (cnt > 10)
bins = pd.cut(cnt[ok], [10, 50, 100, 200, 300, 400, 600]); g = pd.Series(rd[ok]).groupby(bins, observed=True).std(); mid = pd.Series(cnt[ok]).groupby(bins, observed=True).mean()
print(f'{len(pairs)} neighbour pairs without ramps'); print(pd.DataFrame({'vehicles per 5 min': mid.round(0), 'sd of relative difference': g.round(3), 'Poisson would give': (np.sqrt(2 / mid)).round(3)}).to_string())
dq = np.where(freec, (q - pd.DataFrame(q).rolling(5, center=True, min_periods=3).mean().to_numpy()) / q, np.nan)      # cell noise proxies: deviation from a 25-min local mean
m = np.isfinite(dq) & np.isfinite(dev); print(f'corr(speed noise, flow noise) in the same cell: {np.corrcoef(dq[m], dev[m])[0, 1]:+.3f}')
r = dev[:, 50]; r = r[np.isfinite(r)]; r = r - r.mean(); ps = np.abs(np.fft.rfft(r)) ** 2; f = np.fft.rfftfreq(len(r), 5)   # cycles per minute
band = lambda lo, hi: ps[(f >= 1 / hi) & (f < 1 / lo)].sum() / ps[1:].sum()
print(f'speed-residual spectrum of one link, share of variance at periods > 2 h / 30 min–2 h / 10–30 min: {band(120, 1e9):.2f} / {band(30, 120):.2f} / {band(10, 30):.2f}   (white noise: 0.08 / 0.25 / 0.67)')

## 8. There are no demand pulses: within a day the demand is a smooth curve plus white noise

Plot dq/dt of the whole corridor as an image and vertical stripes appear, and we read them, twice, as demand pulses with a 45–55 min
period that returned every weekday (correlation 0.94). Both readings were artefacts of the filter: the full-day gradient is dominated
by the rush-hour slopes that every day shares, and box-detrending a derivative manufactures a broad peak near the box length. The
proper test is a band-pass to periods of 10–60 minutes over the full day, evaluated away from its edges: a link's band-passed flow
correlates **0.03** with the mean of the other 106 links, 1 % of a cell's variance in that band is common to the corridor (the
corridor mean has sd 16.8 veh/h/lane where 107 independent links would give 10.3: a shared wobble of ~13 veh/h/lane against a cell
noise of 107), the corridor-mean pattern correlates **0.02** between days (0.33 at best), and the spectrum in the band is the one of
white noise. On a real
road platoons, signal cycles upstream and incidents put structure at these periods; this generator puts none. The anomaly is an
absence: between the day shape and the per-cell noise there is nothing — which is why every pulse, travel-time and Fourier feature
scored zero, and why the same-slot profile is the whole of the predictable part.

In [ ]:
def bandpass(a, lo=10, hi=60):                       # keep periods between lo and hi minutes (5-min sampling); NaN → 0 after demeaning
    a = np.nan_to_num(a - np.nanmean(a)); F = np.fft.rfft(a); f = np.fft.rfftfreq(len(a), 5); F[(f < 1 / hi) | (f > 1 / lo)] = 0; return np.fft.irfft(F, len(a))
days = np.unique(DAY); win = (np.arange(288) / 12 >= 7) & (np.arange(288) / 12 <= 21); q1 = q / LANES[None, :]
P = np.array([[bandpass(q1[DAY == d][:, k])[win] for k in range(L)] for d in days]).transpose(0, 2, 1)       # (day, step, link): band-pass over the full day, evaluated 7–21 h (away from the filter edges)
cm_ = P.mean(2); cors = [np.corrcoef(P[:, :, k].ravel(), np.delete(P, k, 2).mean(2).ravel())[0, 1] for k in range(L)]
print(f"corr of a link's 10–60-min flow fluctuation with the mean of the other 106 links: median {np.median(cors):.2f} (p10 {np.percentile(cors, 10):.2f}, p90 {np.percentile(cors, 90):.2f}); share of a cell's pulse-band variance that is common-mode: {cm_.var() / P.var():.2f}")
print(f'sd of the corridor-mean band: {cm_.std():.1f} veh/h/lane; median cell sd in the band {np.median(P.std((0, 1))):.1f}; if the 107 links were independent the mean would have sd {np.median(P.std((0, 1))) / np.sqrt(L):.1f}')
kin = L - 1; kmid = int(np.argmin(np.abs(X - X.max() / 2))); kout = 0; n = P.shape[1]
lag = lambda a, b: [np.corrcoef(a[:, :n - l].ravel(), b[:, l:].ravel())[0, 1] for l in range(0, 9)]
for lab, ka, kb in [('entrance → middle (24 km)', kin, kmid), ('middle → exit (24 km)', kmid, kout), ('entrance → exit (49 km)', kin, kout)]:
    cc = lag(P[:, :, ka], P[:, :, kb]); print(f'{lab}: corr per lag of 5 min {np.round(cc, 2)} → best lag {5 * int(np.argmax(cc))} min')
C = np.corrcoef(cm_); wk = np.array([not WEEKEND[DAY == d][0] for d in days]); iu = lambda m: m[np.triu_indices(len(m), 1)]
print(f'the same clock-time pulses on other days? corr of the corridor-mean pulse band between days: median {np.median(iu(C)):.2f}; weekday–weekday {np.median(iu(C[np.ix_(wk, wk)])):.2f}; max over all pairs {iu(C).max():.2f}')
ps = np.mean([np.abs(np.fft.rfft(cm_[i] * np.hanning(n))) ** 2 for i in range(len(days))], 0); f = np.fft.rfftfreq(n, 5); per = 1 / f[1:]; sh = ps[1:] / ps[1:].sum()
print('mean spectrum of the corridor-mean pulse band, variance share by period: 40–60 min %.2f, 25–40 min %.2f, 15–25 min %.2f, 10–15 min %.2f (white noise in this band: 0.17 / 0.30 / 0.53 … by bandwidth: %.2f / %.2f / %.2f / %.2f)' % (tuple(sh[(per >= lo) & (per < hi)].sum() for lo, hi in [(40, 60), (25, 40), (15, 25), (10, 15)]) + tuple((1 / lo - 1 / hi) / (1 / 10 - 1 / 60) for lo, hi in [(40, 60), (25, 40), (15, 25), (10, 15)])))
fig, ax = plt.subplots(figsize=(11, 3.5)); hrs = np.arange(288)[win] / 12; [ax.plot(hrs, cm_[i], lw=.8, alpha=.8, label=str(T[DAY == days[i]][0].date())) for i in range(1, 4)]; ax.set_xlabel('hour'); ax.set_ylabel('corridor-mean pulse (veh/h/lane, 10–60 min band)'); ax.legend(); ax.set_title('three consecutive weekdays, corridor-mean flow in the 10–60 min band: noise, no shared pattern'); plt.show()

## 9. A jam has a fixed head, a straight tail, and no conservation inside it

A jam is born at a bottleneck and its head stays there (median drift 0 km/h over 4,497 episodes in the companion notebook); on I-5 the
tail grows upstream in a straight line, on this I-10 day the jam is one block from km 27 to 45 for five hours whose ends jitter by
~3 km (the block is fragmented by unobserved cells) and drift by less than 1 km/h. The front is sharp in space (speed falls 53–62 km/h within 2 km) but smeared in time (the switch of a cell takes ~40 min
of partial values) — the place is exact, the moment is not, which is precisely the Task-2 onset error. Inside the jam the flow is
less conserved than outside: the median link-to-link flow jump is the same 2.3 %, but the 1 % tail is 21 % against 9 % in free flow, and
a fixed sink of ~250 veh/h sits between km 17 and 22 on every day (right panel: a hidden connector that is in no released file). Shock-wave, cell-transmission and Kalman models all scored at or below persistence here.

In [ ]:
jam_day = days[np.argmax([jam[DAY == d].sum() for d in days])]; s = DAY == jam_day; vd = v[s]; jd = jam[s]
idx = np.argsort(X); Xs = X[idx]; edges = np.r_[Xs[0] - .2, (Xs[1:] + Xs[:-1]) / 2, Xs[-1] + .2]
cm = plt.get_cmap('RdYlGn').copy(); cm.set_bad('#d9d9d9'); fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw=dict(width_ratios=[2, 1]))
ax[0].pcolormesh(edges, np.arange(289) / 12, vd[:, idx], cmap=cm, vmin=0, vmax=120, shading='flat'); ax[0].invert_yaxis(); ax[0].set_xlabel('km from the entrance (traffic →)'); ax[0].set_ylabel('hour'); ax[0].set_title(f'{PANEL} {T[s][0].date()}: speed (grey = not observed)')
vfill = V[s].interpolate(limit=3, limit_direction='both').to_numpy(); jds = (vfill < 0.6 * VF[None, :])[:, idx]; c = int(np.argmax(jds.sum(0))); head = np.full(288, np.nan); tail = np.full(288, np.nan)      # unobserved cells carried from time neighbours for the walk only
for i in range(288):                                              # the contiguous jam through the most-jammed link: head = downstream end, tail = upstream end
    if not jds[i, c]: continue
    a = c
    while a + 1 < L and jds[i, a + 1]: a += 1
    b = c
    while b - 1 >= 0 and jds[i, b - 1]: b -= 1
    head[i] = Xs[a]; tail[i] = Xs[b]
hh = np.arange(288) / 12; ok = np.isfinite(head); ax[0].plot(head[ok], hh[ok], 'k.', ms=3, label='head'); ax[0].plot(tail[ok], hh[ok], 'b.', ms=3, label='tail'); ax[0].legend(loc='lower left', fontsize=8)
dh = np.polyfit(hh[ok], head[ok], 1)[0]; dt_ = np.polyfit(hh[ok], tail[ok], 1)[0]
print(f'biggest jam of the month ({ok.sum()} steps, head at km {np.nanmedian(head):.1f}): head drift {dh:+.2f} km/h (sd {np.nanstd(head):.2f} km), tail drift {dt_:+.2f} km/h (sd {np.nanstd(tail):.2f} km)')
qd = q[s]; rel = np.abs(np.diff(qd, axis=1)) / np.maximum(qd[:, 1:], 1); ins = jd[:, 1:] & jd[:, :-1]; fre = (vd[:, 1:] > .8 * VF[None, 1:]) & (vd[:, :-1] > .8 * VF[None, :-1])
print('relative flow jump between neighbouring links, p50 / p90 / p99: inside the jam', np.round(np.nanpercentile(rel[ins], [50, 90, 99]), 3), ' in free flow', np.round(np.nanpercentile(rel[fre], [50, 90, 99]), 3))
ax[1].plot(Xs, np.nanmean(q[free], 0)[idx], label='10–15 h mean'); ax[1].plot(Xs, np.nanmean(q[(HOUR >= 16) & (HOUR < 19)], 0)[idx], label='16–19 h mean'); ax[1].set_xlabel('km from the entrance'); ax[1].set_ylabel('veh/h'); ax[1].set_title('mean flow along the road: a dip at km 17–22 on every day'); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 10. The conservation score is zero for the truth itself under the public topology

Task 3 scores vehicle conservation: the change of the number of vehicles on a link, dN = Δ(k·L) with k = q/v, against the net flux
in − out (+ ramps). With the released topology as the boundary flows, the truth itself scores **E_LWR ≈ 1.00, S_LWR = 0.0**: the
storage change per 5 min (a fraction of a vehicle) is invisible under the flux noise (two flows × 2.8 % ≈ 13 vehicles per step), and
the ramps are not in the balance anyway (section 1). An independent write-up (shuangranse) found the same (E_LWR 1.001). The
organizers score against a hidden `physics_truth` of generator-internal fluxes projected so that the truth is exact, so S_LWR is
only knowable through the leaderboard; our flow-balance correction (section 1 applied to predictions) was worth +0.0001 there.

In [ ]:
k = q / v; Nveh = k * topo.length_km.to_numpy()[None, :]; dN = np.diff(Nveh, axis=0)                                  # vehicles gained per link per 5 min
on_k = np.column_stack([fill(R.reindex(columns=ON[j]).sum(axis=1, min_count=1)).fillna(0).to_numpy() for j in range(L)]); off_k = np.column_stack([fill(R.reindex(columns=OFF[j]).sum(axis=1, min_count=1)).fillna(0).to_numpy() for j in range(L)])
flux = np.full_like(q, np.nan); flux[:, :-1] = (q[:, 1:] - q[:, :-1] + on_k[:, :-1] - off_k[:, :-1]) / 12                                    # inflow from upstream (order k+1) − outflow + ramps, per 5 min
rhs = flux[:-1]; ok = np.isfinite(dN) & np.isfinite(rhs)
E = np.abs(dN[ok] - rhs[ok]).sum() / (np.abs(rhs[ok]).sum() + 1e-9)        # the official score_task3 formula: Σ|dN − rhs| / Σ|rhs|, S = 1 − min(1, E)
print(f'truth vs its own topology balance: E_LWR = {E:.3f} → S_LWR = {max(0, 1 - min(1, E)):.3f}; corr(dN, net flux) = {np.corrcoef(dN[ok], rhs[ok])[0, 1]:+.3f}')
print(f'typical |dN| per link per 5 min {np.nanmedian(np.abs(dN)):.2f} vehicles; typical |net flux| {np.nanmedian(np.abs(rhs)):.1f} vehicles — the signal is 1/{np.nanmedian(np.abs(rhs)) / np.nanmedian(np.abs(dN)):.0f} of the noise')

## 11. The masks are coin flips with a short memory: no outages, nothing worth predicting

Real detectors fail for hours or days, and a missing cell then predicts its neighbours. Here a missing cell makes the next step of the
same link missing with probability **0.31** against a base rate of 0.16 — a one-step memory, mean run 1.4 steps where an independent
coin gives 1.2 — and has no effect at all on the neighbouring link (0.15). The scoring flag of Task 1 (`is_score_eligible`, 77 % of the
cells) is the same kind of coin (0.79 given an eligible time neighbour), and so is Task 2's horizon eligibility. There are no block
outages to learn. We measured the whole cost of masking at ~1 % on speed and 6 % on flow error — all of it the loss of neighbours.

In [ ]:
fs = sorted(glob.glob(str(D / 'validation' / 'mainline_states_masked' / 'mask_regime=R1' / '*.parquet')))[:2]
for f in fs:
    d = pd.read_parquet(f, columns=['timestamp', 'link_id', 'is_missing', 'is_score_eligible']); Mm = d.pivot(index='timestamp', columns='link_id', values='is_missing').reindex(columns=LINKS).to_numpy().astype(bool); El = d.pivot(index='timestamp', columns='link_id', values='is_score_eligible').reindex(columns=LINKS).to_numpy().astype(bool)
    p = Mm.mean(); pt = Mm[1:][Mm[:-1]].mean(); px = Mm[:, 1:][Mm[:, :-1]].mean()
    runs = np.concatenate([np.diff(np.flatnonzero(np.diff(np.r_[0, Mm[:, j], 0])))[::2] for j in range(L)])
    print(f'{Path(f).name}: P(missing) {p:.3f}; P(missing | time neighbour missing) {pt:.3f}; P(missing | space neighbour missing) {px:.3f}; mean missing run {runs.mean():.2f} steps (independent coin: {1 / (1 - p):.2f}); eligible {El.mean():.3f}, P(eligible | time neighbour eligible) {El[1:][El[:-1]].mean():.3f}')

## 12. What it means for a model

| intuition from real roads | feature built | local effect on the state score | anomaly |
|---|---|---|---|
| ramps feed the mainline | on/off-ramp sums, net ramp balance, ramp-day masks | 0 | 1 |
| ramps during the jam | horizon ramp rows in Task 2 | +0.038 IoU (a leak, not used) | 2 |
| what enters arrives 26 min later | upstream flow shifted by distance/speed | 0 | 3 |
| trucks pull the mean speed down | composition features | nothing to fit | 6 |
| the next cell's noise follows the last | residual second stage, 40 × 5 window regressors | +0.0002, dummy wins | 7 |
| demand pulses exist and repeat; Fourier can predict them | pulse features, travel-time shifts, 6-harmonic day shape, spectral filter | −0.0002 … +0.0004 | 7, 8 |
| shock waves move the tail | shock-speed, CTM, Kalman | ≤ persistence | 9 |
| vehicles are conserved | density head, flow-balance blend, daily-sum constraint | +0.0001 … +0.0003 | 1, 10 |
| detectors fail in blocks | mask prediction | a one-step memory, no blocks | 11 |

What did work is exactly what fits a profile-plus-noise machine: the widest possible averaging of each link's own profile (a 41-link
position window alone +0.0016), a low-rank tensor completion of the month itself (+0.003 on the board), and stacking several such
estimators (+0.0005 per step). The remaining margin to the top of the board is in the jam layer of Task 2, where the place is
knowable and the moment is not.

*All numbers in the text come from `NOTA.md`, the append-only measurement log of this campaign; each code cell above recomputes its
number from the competition release alone. Organizer code: [trafficflowbench-public](https://github.com/jacky850/trafficflowbench-public).*